In [ ]:
import pandas as pd 

In [ ]:
df = pd.read_json('../data/민원(콜센터) 질의응답_다산콜센터_대중교통 안내_Training.json')

In [ ]:
df.info()

In [ ]:
df.isna().sum()

In [ ]:
df.head(10)

#### 문제 
1. 일반 행정 데이터와 대중교통 데이터를 로드 
2. 두개의 데이터를 단순 행 결합
3. 데이터의 필터링
    - 고객의 질문에서 즉각적으로 상담사의 답변이 오는 데이터들만 필터 
    - 고객의 질문이 존재 -> 다음 행의 상담사 답변이 존재하는가?
        - 비어있는 구간의 데이터의 형태를 일반화 
        - '' , ' ', '  ', ... :  '' 통일화 하려면? -> 텍스트 중간에 공백은 그대로 유지하고 좌우의 공백을 제거하는 함수 (strip())
4. 질문 중 중복 데이터를 제거 
5. 고객의 질문과 상담사의 답변이 하나의 행이 되도록 작업 
6. 완료가된 DataFrame을 저장 (민원 질의응답(즉답형데이터).csv) --> data의 백업 
5. 질문들을 모아서 토큰화(Komoran.morphs()) , 벡터화(TF-IDF) 작업 
6. 질문 목록 생성 
    - 여권 재발급 신청 방법을 알려주세요
    - 전입 신고가 인터넷으로 가능한가요?
    - 지방세 환급금을 어디서 신청하나요? 
    - 유사한 질문과 답변을 출력 (2개 씩)

In [ ]:
df2 = pd.read_json("../data/민원(콜센터) 질의응답_다산콜센터_일반행정 문의_Training.json")
df2.info()

In [ ]:
# 2개의 데이터를 단순한 행 결합 
# 인덱스를 기준으로 데이터를 필터링 하기 위해서 인덱스를 초기화
total_df = pd.concat([df, df2], axis=0, ignore_index=True)

In [ ]:
total_df.loc[0, ]

In [ ]:
# 비어있는 구간의 텍스트의 구조 확인 
total_df['고객질문(요청)'].value_counts()

In [ ]:
total_df = total_df.map(lambda x : str(x).strip())

In [ ]:
total_df['고객질문(요청)'].value_counts()

In [ ]:
# 조건식 : 현재 행에서 고객질문(요청) 데이터가 '' 같지 않고 다음 행의 상담사답변이 '' 와 같지 않은 경우
flag = (total_df['고객질문(요청)'] != '') & (total_df.shift(-1)['상담사답변'] != '')
# 조건식2 : 현재 행에서 상담사답변이 ''와 같지 않고 전행의 고객질문(요청) 데이터가 ''와 같지 않은 경우
flag2 = (total_df['상담사답변'] != '') & (total_df.shift(1)['고객질문(요청)'] != '')

total_df.loc[flag | flag2, ]

In [ ]:
df3 = total_df.loc[flag, ]
df3['상담사답변'] = total_df.loc[flag2, '상담사답변'].values
df3

In [ ]:
df4 = total_df.loc[flag, ]
df4['상담사답변'] = total_df.shift(-1).loc[flag, '상담사답변'].values
df4

In [ ]:
# 파일로 저장 
df4.to_csv('민원 질의응답(즉답형데이터).csv', index = False)

In [ ]:
df4.to_excel('민원 질의응답(즉답형데이터).xlsx', index = False)

In [ ]:
df4.info()

In [ ]:
# 질문중 중복 질문에 대한 제거 
df4.drop_duplicates('고객질문(요청)', inplace=True)

In [29]:
df4.info()

<class 'pandas.core.frame.DataFrame'>
Index: 18948 entries, 4 to 89300
Data columns (total 15 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   도메인        18948 non-null  object
 1   카테고리       18948 non-null  object
 2   대화셋일련번호    18948 non-null  object
 3   화자         18948 non-null  object
 4   문장번호       18948 non-null  object
 5   고객의도       18948 non-null  object
 6   상담사의도      18948 non-null  object
 7   QA         18948 non-null  object
 8   고객질문(요청)   18948 non-null  object
 9   상담사질문(요청)  18948 non-null  object
 10  고객답변       18948 non-null  object
 11  상담사답변      18948 non-null  object
 12  개체명        18948 non-null  object
 13  용어사전       18948 non-null  object
 14  지식베이스      18948 non-null  object
dtypes: object(15)
memory usage: 2.3+ MB


In [30]:
from konlpy.tag import Komoran
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [31]:
komoran = Komoran()

def tokenize(text):
    return komoran.morphs(text)

vec = TfidfVectorizer(
    tokenizer=tokenize, 
    lowercase=False, 
    ngram_range=(1,1), 
    min_df = 5, 
    max_df=0.8
)

In [32]:
# 고객질문(요청) 데이터를 벡터화 
X = vec.fit_transform(
    df4['고객질문(요청)']
)

c:\Users\ekfla\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\feature_extraction\text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [35]:
new_questions = [
    '여권 재발급 신청 방법을 알려주세요', 
    '전입 신고가 인터넷에서 가능한가요?', 
    '지방세 환급금을 어디서 신청하나요?'
]

In [36]:
test = vec.transform(new_questions)

In [37]:
sims = cosine_similarity(test, X)

In [38]:
sims

array([[0.        , 0.        , 0.03768027, ..., 0.        , 0.        ,
        0.        ],
       [0.10382223, 0.        , 0.08867365, ..., 0.        , 0.        ,
        0.05229653],
       [0.02110115, 0.        , 0.03995857, ..., 0.03382659, 0.0321687 ,
        0.02175141]], shape=(3, 18948))

In [41]:
df4.reset_index(drop=True, inplace=True)

In [42]:
for idx, sim in enumerate(sims):
    question = new_questions[idx]

    sim_idxs = sim.argsort()[::-1]
    for i in sim_idxs[:2]:
        print(f"""
            유사도의 값 : {round(sim[i], 3)}
            고객의 질문 : {question}
            유사 질문 : {df4.loc[i, '고객질문(요청)']}
            답변 : {df4.loc[i, '상담사답변']}
        """)


            유사도의 값 : 0.707
            고객의 질문 : 여권 재발급 신청 방법을 알려주세요
            유사 질문 : 신청방법을 알려주세요.
            답변 : 주민등록상 세대주와 가까운 주민센터 또는 복지로 홈페이지에서 신청가능하세요.
        

            유사도의 값 : 0.655
            고객의 질문 : 여권 재발급 신청 방법을 알려주세요
            유사 질문 : 신청방법 좀 알려주세요?
            답변 : 우선 사이트에 접속하셔서 회원가입을 해주세요. 청소년일경우 공인인증서가 없으면 본인확인절차를 거쳐 회원가입을 하고, 부모님이나 세대주분께서 가입을 하실 경우 공인인증서로 가입할 수 있습니다.
        

            유사도의 값 : 0.62
            고객의 질문 : 전입 신고가 인터넷에서 가능한가요?
            유사 질문 : 인터넷으로도 신고가능한가요?
            답변 : 방문 접수밖에 안됩니다.
        

            유사도의 값 : 0.583
            고객의 질문 : 전입 신고가 인터넷에서 가능한가요?
            유사 질문 : 전입신고는 가서 해야되죠?
            답변 : 방문신고는 신 거주지 동주민센터에서만 가능합니다.
        

            유사도의 값 : 0.76
            고객의 질문 : 지방세 환급금을 어디서 신청하나요?
            유사 질문 : 지방세 환급금 신청은 어떻게 해야하죠?
            답변 : 인터넷에서 접수를 하셔야 합니다
        

            유사도의 값 : 0.566
            고객의 질문 : 지방세 환급금을 어디서 신청하나요?
            유사 질문 : 환급금을 기부할 수도 있나요?
            답변 : 네 환급금을 사회복지공동모